# Notebook 29 — Explicit Universality Boundary Extraction

This notebook takes the control-space universality maps from Notebook 28 and turns them into explicit boundary curves.

Goal:
- identify where the effective-noise projection remains valid,
- extract boundary curves in control space,
- summarize the edge of the projection-stable regime.

We work with the same diagnostics as before:
- lambda shift from baseline,
- axis-slice collapse loss,
- response-curve mismatch.

From these we define a graded universality score and then extract approximate boundary curves in:

1. `(T, alpha)` space
2. `(T, omega_max)` space

This notebook is the direct follow-up to Notebook 28 and is meant to produce the cleanest paper-style statement of where the reduced one-parameter noise geometry is valid.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and control grids

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

T_vals = np.array([20.0, 24.0, 26.0, 30.0, 34.0])
alpha_vals = np.array([0.06, 0.08, 0.10, 0.12, 0.14])
omega_vals = np.array([0.95, 1.02, 1.09, 1.16, 1.23])

print("Baseline:", baseline)
print("T grid:", T_vals)
print("alpha grid:", alpha_vals)
print("omega grid:", omega_vals)


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    H = build_time_dependent_hamiltonian(
        params['T'], params['omega_max'], params['alpha'], V, params['delta0'], params['p'], params['q']
    )
    times = np.linspace(0.0, params['T'], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            vals.append(basis_state.overlap(state))
        else:
            val = basis_state.dag() * state * basis_state
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            score = np.abs(basis_state.overlap(final_state)) ** 2
        else:
            val = basis_state.dag() * final_state * basis_state
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Shared noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 13)
gamma_phi_vals = np.linspace(0.0, 0.12, 13)


## Helper: evaluate one protocol

In [ ]:
def evaluate_protocol(params):
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=140
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        mapped = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    lam = float(fit.x)
    axis_loss = float(fit.fun)

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    gamma_eff_flat = gamma_eff_grid.ravel()
    S_flat = S_norm.ravel()
    order = np.argsort(gamma_eff_flat)
    ge = gamma_eff_flat[order]
    sv = S_flat[order]

    n_bins = 12
    bins = np.linspace(ge.min(), ge.max(), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(n_bins, np.nan)

    for k in range(n_bins):
        mask = (ge >= bins[k]) & (ge < bins[k+1])
        if np.any(mask):
            means[k] = np.mean(sv[mask])

    valid = ~np.isnan(means)
    x = centers[valid]
    y = means[valid]

    return {
        'params': params,
        'S_norm': S_norm,
        'lambda': lam,
        'axis_loss': axis_loss,
        'curve_x': x,
        'curve_y': y,
    }


## Baseline reference

In [ ]:
baseline_res = evaluate_protocol(baseline)
lambda0 = baseline_res['lambda']
curve0_x = baseline_res['curve_x']
curve0_y = baseline_res['curve_y']
interp0 = interp1d(curve0_x, curve0_y, kind='linear', fill_value='extrapolate')

print("baseline lambda =", lambda0)


## Universality score

In [ ]:
def curve_mismatch_to_baseline(x, y):
    y_ref = interp0(np.clip(x, curve0_x.min(), curve0_x.max()))
    return float(np.mean((y - y_ref) ** 2))

def universality_score(lambda_shift, axis_loss, curve_mismatch,
                       lambda_scale=0.15, loss_scale=0.001, curve_scale=0.001):
    score = np.exp(
        -(lambda_shift / lambda_scale)
        -(axis_loss / loss_scale)
        -(curve_mismatch / curve_scale)
    )
    return float(score)

boundary_level = 0.30
print("boundary level =", boundary_level)


## Evaluate T × alpha grid

In [ ]:
TA_score = np.zeros((len(alpha_vals), len(T_vals)))
TA_lambda = np.zeros((len(alpha_vals), len(T_vals)))
TA_curve = np.zeros((len(alpha_vals), len(T_vals)))

for i, alpha in enumerate(alpha_vals):
    for j, T in enumerate(T_vals):
        params = {**baseline, 'T': float(T), 'alpha': float(alpha)}
        res = evaluate_protocol(params)

        lambda_shift = abs(res['lambda'] - lambda0)
        curve_mismatch = curve_mismatch_to_baseline(res['curve_x'], res['curve_y'])
        score = universality_score(lambda_shift, res['axis_loss'], curve_mismatch)

        TA_score[i, j] = score
        TA_lambda[i, j] = res['lambda']
        TA_curve[i, j] = curve_mismatch

        print(f"T-alpha done: T={T}, alpha={alpha}, score={score:.3f}")


## Evaluate T × omega_max grid

In [ ]:
TO_score = np.zeros((len(omega_vals), len(T_vals)))
TO_lambda = np.zeros((len(omega_vals), len(T_vals)))
TO_curve = np.zeros((len(omega_vals), len(T_vals)))

for i, omega in enumerate(omega_vals):
    for j, T in enumerate(T_vals):
        params = {**baseline, 'T': float(T), 'omega_max': float(omega)}
        res = evaluate_protocol(params)

        lambda_shift = abs(res['lambda'] - lambda0)
        curve_mismatch = curve_mismatch_to_baseline(res['curve_x'], res['curve_y'])
        score = universality_score(lambda_shift, res['axis_loss'], curve_mismatch)

        TO_score[i, j] = score
        TO_lambda[i, j] = res['lambda']
        TO_curve[i, j] = curve_mismatch

        print(f"T-omega done: T={T}, omega={omega}, score={score:.3f}")


## Boundary extraction helper

In [ ]:
def extract_vertical_boundary(score_map, xvals, yvals, level):
    # For each y row, find first x where score crosses downward through level.
    boundary_x = []
    boundary_y = []

    for i, y in enumerate(yvals):
        row = score_map[i, :]
        crossing = None
        for j in range(len(xvals) - 1):
            v0, v1 = row[j], row[j + 1]
            if (v0 >= level and v1 < level) or (v0 > level and v1 <= level):
                x0, x1 = xvals[j], xvals[j + 1]
                if v1 == v0:
                    xc = x0
                else:
                    xc = x0 + (level - v0) * (x1 - x0) / (v1 - v0)
                crossing = xc
                break

        if crossing is None:
            # if entire row above level, set to max x; if entire row below, skip
            if np.max(row) >= level and np.min(row) >= level:
                crossing = float(xvals[-1])

        if crossing is not None:
            boundary_x.append(float(crossing))
            boundary_y.append(float(y))

    return np.array(boundary_x), np.array(boundary_y)


## Extract boundary curves

In [ ]:
TA_bx, TA_by = extract_vertical_boundary(TA_score, T_vals, alpha_vals, boundary_level)
TO_bx, TO_by = extract_vertical_boundary(TO_score, T_vals, omega_vals, boundary_level)

print("T-alpha boundary points:", list(zip(TA_bx, TA_by)))
print("T-omega boundary points:", list(zip(TO_bx, TO_by)))


## T × alpha score and boundary

In [ ]:
plt.figure(figsize=(7.4, 5.6))
im = plt.imshow(
    TA_score,
    origin='lower',
    aspect='auto',
    extent=[T_vals[0], T_vals[-1], alpha_vals[0], alpha_vals[-1]],
    vmin=0, vmax=1
)
plt.contour(T_vals, alpha_vals, TA_score, levels=[boundary_level], colors='white', linewidths=1.3)
if len(TA_bx) > 0:
    plt.plot(TA_bx, TA_by, color='red', linewidth=2.0, label='extracted boundary')
plt.xlabel('T')
plt.ylabel('alpha')
plt.title('T × alpha universality boundary')
plt.colorbar(im, label='universality score')
if len(TA_bx) > 0:
    plt.legend()
plt.tight_layout()
plt.show()


## T × omega score and boundary

In [ ]:
plt.figure(figsize=(7.4, 5.6))
im = plt.imshow(
    TO_score,
    origin='lower',
    aspect='auto',
    extent=[T_vals[0], T_vals[-1], omega_vals[0], omega_vals[-1]],
    vmin=0, vmax=1
)
plt.contour(T_vals, omega_vals, TO_score, levels=[boundary_level], colors='white', linewidths=1.3)
if len(TO_bx) > 0:
    plt.plot(TO_bx, TO_by, color='red', linewidth=2.0, label='extracted boundary')
plt.xlabel('T')
plt.ylabel('omega_max')
plt.title('T × omega universality boundary')
plt.colorbar(im, label='universality score')
if len(TO_bx) > 0:
    plt.legend()
plt.tight_layout()
plt.show()


## Boundary curves as standalone figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

if len(TA_bx) > 0:
    axes[0].plot(TA_by, TA_bx, marker='o')
axes[0].axvline(baseline['T'], linestyle='--', alpha=0.7)
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('boundary T')
axes[0].set_title('Extracted T boundary vs alpha')

if len(TO_bx) > 0:
    axes[1].plot(TO_by, TO_bx, marker='o')
axes[1].axvline(baseline['T'], linestyle='--', alpha=0.7)
axes[1].set_xlabel('omega_max')
axes[1].set_ylabel('boundary T')
axes[1].set_title('Extracted T boundary vs omega_max')

plt.tight_layout()
plt.show()


## Compare raw diagnostics near the boundary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))

im0 = axes[0,0].imshow(
    TA_lambda, origin='lower', aspect='auto',
    extent=[T_vals[0], T_vals[-1], alpha_vals[0], alpha_vals[-1]]
)
axes[0,0].set_title('T × alpha fitted lambda')
axes[0,0].set_xlabel('T')
axes[0,0].set_ylabel('alpha')
if len(TA_bx) > 0:
    axes[0,0].plot(TA_bx, TA_by, color='white', linewidth=1.8)

im1 = axes[0,1].imshow(
    TA_curve, origin='lower', aspect='auto',
    extent=[T_vals[0], T_vals[-1], alpha_vals[0], alpha_vals[-1]]
)
axes[0,1].set_title('T × alpha curve mismatch')
axes[0,1].set_xlabel('T')
axes[0,1].set_ylabel('alpha')
if len(TA_bx) > 0:
    axes[0,1].plot(TA_bx, TA_by, color='white', linewidth=1.8)

im2 = axes[1,0].imshow(
    TO_lambda, origin='lower', aspect='auto',
    extent=[T_vals[0], T_vals[-1], omega_vals[0], omega_vals[-1]]
)
axes[1,0].set_title('T × omega fitted lambda')
axes[1,0].set_xlabel('T')
axes[1,0].set_ylabel('omega_max')
if len(TO_bx) > 0:
    axes[1,0].plot(TO_bx, TO_by, color='white', linewidth=1.8)

im3 = axes[1,1].imshow(
    TO_curve, origin='lower', aspect='auto',
    extent=[T_vals[0], T_vals[-1], omega_vals[0], omega_vals[-1]]
)
axes[1,1].set_title('T × omega curve mismatch')
axes[1,1].set_xlabel('T')
axes[1,1].set_ylabel('omega_max')
if len(TO_bx) > 0:
    axes[1,1].plot(TO_bx, TO_by, color='white', linewidth=1.8)

fig.colorbar(im0, ax=axes[0,0], label='lambda')
fig.colorbar(im1, ax=axes[0,1], label='mismatch')
fig.colorbar(im2, ax=axes[1,0], label='lambda')
fig.colorbar(im3, ax=axes[1,1], label='mismatch')
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Explicit universality boundary summary")
lines.append("")
lines.append(f"Baseline lambda = {lambda0:.4f}")
lines.append(f"Boundary score level = {boundary_level:.2f}")
lines.append("")

if len(TA_bx) > 0:
    lines.append("T × alpha extracted boundary:")
    for x, y in zip(TA_bx, TA_by):
        lines.append(f"- alpha={y:.3f} -> boundary T≈{x:.3f}")
else:
    lines.append("No extracted T × alpha boundary found.")

lines.append("")

if len(TO_bx) > 0:
    lines.append("T × omega extracted boundary:")
    for x, y in zip(TO_bx, TO_by):
        lines.append(f"- omega_max={y:.3f} -> boundary T≈{x:.3f}")
else:
    lines.append("No extracted T × omega boundary found.")

lines.append("")
lines.append("Interpretation:")
lines.append("- The extracted curves mark where the projection-stable regime stops matching the baseline effective-noise geometry.")
lines.append("- Inside the boundary, the reduced one-parameter description remains accurate.")
lines.append("- Outside the boundary, lambda drift and response-curve distortion become too large for a single shared geometry to remain trustworthy.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook turns the universality score from Notebook 28 into explicit boundary curves.

That is the cleanest control-space result in the series:

- a reduced effective-noise geometry exists,
- it remains valid only inside a bounded region of control space,
- and the edge of that region can be extracted as a concrete curve rather than described only qualitatively.

So the final statement becomes:

**the phase-locked CZ protocol family admits an explicit universality boundary in control space, separating a projection-stable regime from a regime where the effective one-parameter noise description breaks down.**
